In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import wandb
from pathlib import Path

from models.motionstream import create_motionstream, create_motionstream_lite
from utils.datasets import MotionAnomalyDataset, create_dataloaders
from utils.metrics import AnomalyMetrics, count_parameters, compute_inference_speed
from utils.training import Trainer, create_optimizer, create_scheduler
from utils.visualization import plot_training_curves, visualize_optical_flow
from utils.export import export_to_onnx

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Load Configuration

In [ ]:
# Load training config
with open('../configs/motionstream.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration:")
print(yaml.dump(config, default_flow_style=False))

## 2. Prepare Dataset

In [ ]:
# Create datasets
train_dataset = MotionAnomalyDataset(
    root=config['dataset']['root'],
    split='train',
    num_frames=config['model']['num_frames'],
    frame_size=config['model']['frame_size'],
    compute_flow=True
)

val_dataset = MotionAnomalyDataset(
    root=config['dataset']['root'],
    split='val',
    num_frames=config['model']['num_frames'],
    frame_size=config['model']['frame_size'],
    compute_flow=True
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Anomaly ratio (train): {train_dataset.get_anomaly_ratio():.2%}")

# Create dataloaders
train_loader = create_dataloaders(
    train_dataset,
    batch_size=config['training']['batch_size'],
    num_workers=config['dataset']['num_workers'],
    split='train'
)

val_loader = create_dataloaders(
    val_dataset,
    batch_size=config['validation']['batch_size'],
    num_workers=config['dataset']['num_workers'],
    split='val'
)

In [ ]:
# Visualize sample optical flow
sample_flow, sample_label = train_dataset[0]
print(f"Optical flow shape: {sample_flow.shape}")  # (T, 2, H, W)
print(f"Label: {'Anomaly' if sample_label == 1 else 'Normal'}")

# Visualize first frame
visualize_optical_flow(
    sample_flow[0].numpy(),  # (2, H, W)
    save_path='optical_flow_sample.png'
)
plt.show()

## 3. Create Model

In [ ]:
# Create model (use lite version for faster training/inference)
use_lite = config['model'].get('use_lite', True)

if use_lite:
    model = create_motionstream_lite(pretrained=False)
    print("Using MotionStream Lite (3D CNN)")
else:
    model = create_motionstream(pretrained=False)
    print("Using MotionStream Full (CNN+LSTM)")

model = model.to(device)

# Model info
param_info = count_parameters(model)
print(f"\nModel Parameters:")
print(f"  Total: {param_info['total_params']:,}")
print(f"  Trainable: {param_info['trainable_params']:,}")

# Test forward pass
dummy_input = torch.randn(2, 8, 2, 224, 224).to(device)
dummy_output = model(dummy_input)
print(f"\nInput shape: {dummy_input.shape}")
print(f"Output shape: {dummy_output.shape}")

## 4. Setup Training

In [ ]:
# Loss function with positive weight for class imbalance
pos_weight = torch.tensor([config['training']['loss'].get('pos_weight', 3.0)]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Optimizer
optimizer = create_optimizer(
    model,
    optimizer_type=config['training']['optimizer']['type'],
    lr=config['training']['optimizer']['lr'],
    weight_decay=config['training']['optimizer']['weight_decay']
)

# Scheduler
scheduler = create_scheduler(
    optimizer,
    scheduler_type=config['training']['scheduler']['type'],
    num_epochs=config['training']['num_epochs'],
    patience=config['training']['scheduler'].get('patience', 10)
)

# Metrics tracker
metrics_tracker = AnomalyMetrics()

print("Training setup complete!")

## 5. Initialize Weights & Biases (Optional)

In [ ]:
# Initialize W&B
use_wandb = config['wandb']['enabled']

if use_wandb:
    wandb.init(
        project=config['wandb']['project'],
        entity=config['wandb']['entity'],
        name=config['wandb']['name'],
        tags=config['wandb']['tags'],
        config=config
    )
    wandb.watch(model, log='all', log_freq=100)
    print("W&B initialized!")
else:
    print("W&B disabled")

## 6. Train Model

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    use_wandb=use_wandb
)

# Train
trainer.fit(
    num_epochs=config['training']['num_epochs'],
    metric_fn=metrics_tracker,
    save_best=config['checkpoints']['save_best'],
    checkpoint_dir=config['checkpoints']['save_dir']
)

## 7. Visualize Results

In [ ]:
# Plot training curves
train_losses = trainer.train_losses
val_losses = trainer.val_losses
train_aucs = [m.get('auc_roc', 0) for m in trainer.train_metrics]
val_aucs = [m.get('auc_roc', 0) for m in trainer.val_metrics]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0].plot(val_losses, label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# AUC-ROC curves
axes[1].plot(train_aucs, label='Train AUC-ROC', linewidth=2)
axes[1].plot(val_aucs, label='Val AUC-ROC', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC-ROC')
axes[1].set_title('AUC-ROC Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('motionstream_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Load best model
checkpoint_path = Path(config['checkpoints']['save_dir']) / 'best_model.pth'
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"Best model (Epoch {checkpoint['epoch']}):")
print(f"  Train Loss: {checkpoint['train_loss']:.4f}")
print(f"  Val Loss: {checkpoint['val_loss']:.4f}")

In [ ]:
# Evaluate on validation set
metrics_tracker.reset()
val_results = trainer.validate(metric_fn=metrics_tracker)

print("\nValidation Results:")
print(f"  AUC-ROC: {val_results.get('auc_roc', 0):.4f}")
print(f"  Optimal Threshold: {val_results.get('optimal_threshold', 0.5):.4f}")
print(f"  Accuracy: {val_results.get('accuracy', 0):.4f}")
print(f"  Precision: {val_results.get('precision', 0):.4f}")
print(f"  Recall: {val_results.get('recall', 0):.4f}")
print(f"  F1-Score: {val_results.get('f1', 0):.4f}")

In [ ]:
# Plot ROC curve
metrics_tracker.plot_roc_curve(save_path='motionstream_roc_curve.png')
plt.show()

## 8. Benchmark Inference Speed

In [ ]:
# Benchmark PyTorch model
benchmark_results = compute_inference_speed(
    model=model,
    input_shape=(1, 8, 2, 224, 224),
    device=device,
    num_iterations=1000
)

print("\nInference Speed (PyTorch):")
print(f"  Avg time: {benchmark_results['avg_inference_ms']:.2f} ms")
print(f"  Throughput: {benchmark_results['fps']:.1f} FPS")

## 9. Export to ONNX

In [ ]:
# Export to ONNX
onnx_output_dir = '../../services/models/motionstream/weights'
Path(onnx_output_dir).mkdir(parents=True, exist_ok=True)

model_name = 'motionstream_lite.onnx' if use_lite else 'motionstream.onnx'
onnx_path = str(Path(onnx_output_dir) / model_name)

success = export_to_onnx(
    model=model,
    input_shape=(1, 8, 2, 224, 224),
    output_path=onnx_path,
    opset_version=14,
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    input_names=['input'],
    output_names=['output']
)

if success:
    print(f"\n✓ ONNX model exported to: {onnx_path}")
else:
    print("\n✗ ONNX export failed")

In [ ]:
# Validate ONNX model
from utils.export import validate_onnx_model, benchmark_onnx_model

validation_success = validate_onnx_model(
    onnx_path=onnx_path,
    pytorch_model=model,
    input_shape=(1, 8, 2, 224, 224),
    tolerance=1e-5
)

if validation_success:
    print("\n✓ ONNX model validation passed")

# Benchmark ONNX Runtime
onnx_benchmark = benchmark_onnx_model(
    onnx_path=onnx_path,
    input_shape=(1, 8, 2, 224, 224),
    num_iterations=1000,
    provider='CPUExecutionProvider'
)

print("\nInference Speed Comparison:")
print(f"  PyTorch: {benchmark_results['avg_inference_ms']:.2f} ms")
print(f"  ONNX Runtime: {onnx_benchmark['avg_inference_ms']:.2f} ms")
speedup = benchmark_results['avg_inference_ms'] / onnx_benchmark['avg_inference_ms']
print(f"  Speedup: {speedup:.2f}x")

## 10. Test on Sample Data

In [ ]:
# Test inference
model.eval()
optimal_threshold = val_results.get('optimal_threshold', 0.5)

# Get a batch from validation set
test_samples, test_labels = next(iter(val_loader))
test_samples = test_samples.to(device)

# Predict
with torch.no_grad():
    outputs = model(test_samples)
    probs = torch.sigmoid(outputs)
    preds = (probs > optimal_threshold).long()

# Display results
print("\nSample Predictions:")
for i in range(min(5, len(test_samples))):
    pred_label = 'Anomaly' if preds[i] == 1 else 'Normal'
    true_label = 'Anomaly' if test_labels[i] == 1 else 'Normal'
    confidence = probs[i].item()
    
    status = "✓" if pred_label == true_label else "✗"
    print(f"  {status} Predicted: {pred_label} ({confidence:.2%}) | True: {true_label}")

## Summary

✅ **Training Complete!**

**Next Steps:**
1. Deploy model: `python ../scripts/deploy_models.py --model motionstream`
2. Restart services: `docker-compose restart motionstream`
3. Monitor inference: `docker-compose logs -f motionstream`

**Model Artifacts:**
- Checkpoint: `./checkpoints/motionstream/best_model.pth`
- ONNX: `../../services/models/motionstream/weights/motionstream_lite.onnx`
- Metrics: Logged to W&B (if enabled)

**Performance:**
- AUC-ROC: Check val_results above
- Optimal Threshold: Check val_results above
- Inference: Check onnx_benchmark above